# Spider ET Variation Boostfactor Determination

This notebook computes the uncorrected boostfactor from the transversely integrated ET data in `data/ET_results_variations.npz`. It fits the shared full ET dataset once, fits every reduced configuration, and ranks the configurations by both MADBead fit quality and reconstruction of the full boostfactor curve.

In [ ]:
using Statistics
using LinearAlgebra
using LsqFit
using LaTeXStrings
using Plots
using Plots.Measures
using DataFrames
using Distributions
using Measurements
using NPZ
using JLD2

const SPIDER_REPO = begin
    cwd = pwd()
    if basename(cwd) == "spider-bead-pull" && isfile(joinpath(cwd, "data", "ET_results_variations.npz"))
        cwd
    elseif isfile(joinpath(cwd, "spider-bead-pull", "data", "ET_results_variations.npz"))
        joinpath(cwd, "spider-bead-pull")
    else
        error("Run this notebook from the workspace root or from the spider-bead-pull directory.")
    end
end

const MADMAX_ROOT = normpath(joinpath(SPIDER_REPO, ".."))
include(joinpath(MADMAX_ROOT, "MADBead", "MADBead.jl"));

const DATA_FILE = joinpath(SPIDER_REPO, "data", "ET_results_variations.npz")
const RESULTS_FILE = joinpath(SPIDER_REPO, "data", "5_result_bf_determination_spider_ET_variations.jld2")
const CONFIGURATION_LOOKUP_COLUMNS = ["configuration_index", "line_count", "z_slice_count", "gap_count"]

notebook_fig_dir = joinpath(SPIDER_REPO, "figs", "comparisons", "spider_et_boostfactor_variations")
mkpath(notebook_fig_dir)

function save_notebook_plot(plt, stem; formats=("png", "svg"))
    for ext in formats
        savefig(plt, joinpath(notebook_fig_dir, "$(stem).$(ext)"))
    end
    return plt
end

function config_slug(config)
    line = lpad(string(Int(config.line_count)), 2, "0")
    zslice = lpad(string(Int(config.z_slice_count)), 2, "0")
    gap = lpad(string(Int(config.gap_count)), 2, "0")
    return joinpath("lines_$(line)", "zslices_$(zslice)", "gaps_$(gap)")
end

function config_fig_dir(config)
    dir = joinpath(SPIDER_REPO, "figs", "configs", config_slug(config))
    mkpath(dir)
    return dir
end

function save_config_plot(plt, config, stem; formats=("png", "svg"))
    dir = config_fig_dir(config)
    paths = String[]
    for ext in formats
        path = joinpath(dir, "$(stem).$(ext)")
        savefig(plt, path)
        push!(paths, path)
    end
    return paths
end

## Load ET Variation Data

The variation file stores one reduced ET dataset per configuration, plus one shared full ET dataset. The reduced z positions are relative distances left of the mirror and map to absolute positions as $z_m - z_{rel}$.

In [ ]:
const REQUIRED_VARIATION_KEYS = [
    "ETs_reduced",
    "z_reduced",
    "z_count",
    "ETs_full",
    "z_full",
    "frequencies",
    "configuration_indices",
    "configuration_lookup",
    "z_ixs_used",
    "z_ixs_used_count",
]

function assert_finite_vector(name, values)
    @assert ndims(values) == 1 "$(name) must be a vector."
    @assert !isempty(values) "$(name) must not be empty."
    @assert all(isfinite.(values)) "$(name) contains non-finite values."
    return values
end

function orient_ET(name, ET_zf, ν, z_rel)
    @assert ndims(ET_zf) == 2 "$(name) must be a 2D array with shape (z, frequency)."
    @assert size(ET_zf, 1) == length(z_rel) "$(name) z dimension does not match its z array."
    @assert size(ET_zf, 2) == length(ν) "$(name) frequency dimension does not match frequencies."
    finite_mask = isfinite.(real.(ET_zf)) .& isfinite.(imag.(ET_zf))
    bad_count = count(.!finite_mask)
    if bad_count > 0
        @warn "$(name) contains $(bad_count) non-finite ET value(s); cleanup will skip affected reduced frequencies."
    end
    return permutedims(ET_zf, (2, 1))
end

function make_ET_measurements(ET_fz; rel_err=0.05)
    @assert 0 <= rel_err < 1 "rel_err should be a fractional uncertainty."
    real_err = rel_err .* abs.(real.(ET_fz))
    imag_err = rel_err .* abs.(imag.(ET_fz))
    return (real.(ET_fz) .± real_err) .+ 1im .* (imag.(ET_fz) .± imag_err)
end

function build_dataset(label, ET_zf, z_rel, ν; z_mirror, rel_err)
    z_rel = vec(Float64.(z_rel))
    ET_zf = Array{ComplexF64}(ET_zf)
    ET_fz = orient_ET(label, ET_zf, ν, z_rel)
    z_abs = z_mirror .- z_rel
    @assert all(z_abs .< z_mirror) "$(label) z values must lie left of the mirror."

    order = sortperm(z_abs)
    ET_fz = ET_fz[:, order]
    z_rel = z_rel[order]
    z_abs = z_abs[order]

    return (
        label=label,
        z_rel=z_rel,
        z_abs=z_abs,
        ET=ET_fz,
        ET_meas=make_ET_measurements(ET_fz; rel_err=rel_err),
    )
end

function configuration_label(configuration_index, line_count, z_slice_count, gap_count)
    return "cfg$(configuration_index)_L$(line_count)_Z$(z_slice_count)_G$(gap_count)"
end

function config_z_ixs_used(z_ixs_pad, z_ixs_count, row)
    if ndims(z_ixs_pad) == 2
        counts = vec(z_ixs_count)
        @assert size(z_ixs_pad, 1) == length(counts) "z_ixs_used row count mismatch."
        n = counts[row]
        @assert 0 <= n <= size(z_ixs_pad, 2) "z_ixs_used_count contains an invalid value at row $(row)."
        return n == 0 ? Int[] : collect(vec(z_ixs_pad[row, 1:n]))
    elseif ndims(z_ixs_pad) == 3
        @assert ndims(z_ixs_count) == 2 "3D z_ixs_used requires 2D z_ixs_used_count."
        @assert size(z_ixs_pad, 1) == size(z_ixs_count, 1) "z_ixs_used configuration count mismatch."
        @assert size(z_ixs_pad, 2) == size(z_ixs_count, 2) "z_ixs_used gap count mismatch."
        z_ixs = Int[]
        for gap in axes(z_ixs_count, 2)
            n = z_ixs_count[row, gap]
            @assert 0 <= n <= size(z_ixs_pad, 3) "z_ixs_used_count contains an invalid value at row $(row), gap $(gap)."
            if n > 0
                append!(z_ixs, vec(z_ixs_pad[row, gap, 1:n]))
            end
        end
        return collect(z_ixs[z_ixs .>= 0])
    else
        error("z_ixs_used must be either 2D or 3D.")
    end
end

function load_ET_variation_results(data_file; z_mirror, rel_err=0.05)
    @assert isfile(data_file) "Missing ET variation data file: $(data_file)"
    data = npzread(data_file)
    for key in REQUIRED_VARIATION_KEYS
        @assert haskey(data, key) "Missing key $(key) in $(data_file)."
    end

    ν = assert_finite_vector("frequencies", vec(Float64.(data["frequencies"])))
    @assert issorted(ν) "frequencies must be sorted in ascending order."

    z_full = assert_finite_vector("z_full", vec(Float64.(data["z_full"])))
    ET_full = Array{ComplexF64}(data["ETs_full"])
    @assert size(ET_full) == (length(z_full), length(ν)) "ETs_full must have shape (z_full, frequency)."
    full = build_dataset("full", ET_full, z_full, ν; z_mirror=z_mirror, rel_err=rel_err)

    ET_pad = ComplexF64.(data["ETs_reduced"])
    z_pad = Float64.(data["z_reduced"])
    z_count = Int.(vec(data["z_count"]))
    z_ixs_pad = Int.(data["z_ixs_used"])
    z_ixs_count = Int.(data["z_ixs_used_count"])
    lookup = Int.(data["configuration_lookup"])
    indices = Int.(vec(data["configuration_indices"]))

    @assert ndims(ET_pad) == 3 "ETs_reduced must have shape (configuration, max_z, frequency)."
    @assert ndims(z_pad) == 2 "z_reduced must have shape (configuration, max_z)."
    @assert size(lookup, 2) == length(CONFIGURATION_LOOKUP_COLUMNS) "Unexpected configuration lookup shape."
    @assert size(lookup, 1) == size(ET_pad, 1) == size(z_pad, 1) == length(z_count) == length(indices) "Configuration counts do not agree."
    @assert size(ET_pad, 2) == size(z_pad, 2) "Reduced ET and z padding dimensions do not agree."
    @assert size(ET_pad, 3) == length(ν) "Padded reduced ET frequency dimension mismatch."
    @assert all(0 .< z_count .<= size(ET_pad, 2)) "z_count contains invalid values."
    @assert all(z_ixs_count .>= 0) "z_ixs_used_count contains invalid values."
    @assert size(z_ixs_pad, 1) == size(ET_pad, 1) "z_ixs_used configuration count mismatch."

    reduced_configurations = NamedTuple[]
    for row in axes(lookup, 1)
        configuration_index, line_count, z_slice_count, gap_count = lookup[row, :]
        @assert configuration_index == indices[row] "configuration_indices and configuration_lookup disagree at row $(row)."
        n_z = z_count[row]
        ET_zf = ET_pad[row, 1:n_z, :]
        z_rel = vec(z_pad[row, 1:n_z])
        @assert all(isfinite.(z_rel)) "Configuration $(configuration_index) contains non-finite z values."
        z_ixs_used = config_z_ixs_used(z_ixs_pad, z_ixs_count, row)
        label = configuration_label(configuration_index, line_count, z_slice_count, gap_count)
        dataset = build_dataset(label, ET_zf, z_rel, ν; z_mirror=z_mirror, rel_err=rel_err)
        push!(
            reduced_configurations,
            merge(
                dataset,
                (
                    configuration_index=configuration_index,
                    line_count=line_count,
                    z_slice_count=z_slice_count,
                    gap_count=gap_count,
                    lookup_row=row,
                    z_ixs_used=collect(z_ixs_used),
                ),
            ),
        )
    end

    return ν, reduced_configurations, full, lookup
end

function robust_median_mad(values)
    finite_values = values[isfinite.(values)]
    if isempty(finite_values)
        return 0.0, 0.0
    end
    med = median(finite_values)
    scale = 1.4826 * median(abs.(finite_values .- med))
    return med, scale
end

function hampel_component_flags(values; window=51, threshold=12.0)
    @assert isodd(window) "Hampel window must be odd."
    @assert window >= 3 "Hampel window must contain at least 3 points."
    x = Float64.(values)
    n = length(x)
    flags = falses(n)
    scores = zeros(Float64, n)

    global_med, global_scale = robust_median_mad(x)
    half_window = window ÷ 2

    for i in eachindex(x)
        if !isfinite(x[i])
            flags[i] = true
            scores[i] = Inf
            continue
        end

        lo = max(firstindex(x), i - half_window)
        hi = min(lastindex(x), i + half_window)
        neighbor_idx = vcat(collect(lo:i-1), collect(i+1:hi))
        neighbors = x[neighbor_idx]
        finite_neighbors = neighbors[isfinite.(neighbors)]

        local_med, local_scale = isempty(finite_neighbors) ? (global_med, global_scale) : robust_median_mad(finite_neighbors)
        scale = max(local_scale, global_scale, eps(Float64))
        scores[i] = abs(x[i] - local_med) / scale
        flags[i] = scores[i] > threshold
    end

    return flags, scores
end

function detect_reduced_ET_outliers(dataset, ν; window=51, threshold=12.0)
    @assert size(dataset.ET, 1) == length(ν) "Reduced ET frequency dimension must match ν."

    point_mask = falses(size(dataset.ET))
    real_score = zeros(Float64, size(dataset.ET))
    imag_score = zeros(Float64, size(dataset.ET))

    for z_idx in axes(dataset.ET, 2)
        real_flags, real_scores = hampel_component_flags(real.(dataset.ET[:, z_idx]); window=window, threshold=threshold)
        imag_flags, imag_scores = hampel_component_flags(imag.(dataset.ET[:, z_idx]); window=window, threshold=threshold)
        point_mask[:, z_idx] .= real_flags .| imag_flags
        real_score[:, z_idx] .= real_scores
        imag_score[:, z_idx] .= imag_scores
    end

    frequency_skip_mask = vec(any(point_mask, dims=2))
    keep_frequency_mask = .!frequency_skip_mask

    return (
        point_mask=point_mask,
        real_score=real_score,
        imag_score=imag_score,
        frequency_skip_mask=frequency_skip_mask,
        keep_frequency_mask=keep_frequency_mask,
        skipped_frequencies=ν[frequency_skip_mask],
        window=window,
        threshold=threshold,
    )
end

function apply_frequency_mask(values::AbstractVector, keep_frequency_mask)
    @assert length(values) == length(keep_frequency_mask) "Frequency mask length mismatch."
    return values[keep_frequency_mask]
end

function apply_frequency_mask(dataset::NamedTuple, keep_frequency_mask; label_suffix="clean")
    @assert size(dataset.ET, 1) == length(keep_frequency_mask) "Dataset frequency dimension does not match mask."
    return merge(
        dataset,
        (
            label="$(dataset.label)_$(label_suffix)",
            ET=dataset.ET[keep_frequency_mask, :],
            ET_meas=dataset.ET_meas[keep_frequency_mask, :],
        ),
    )
end

function outlier_summary_table(configurations, outliers, ν)
    df = DataFrame(
        configuration_index=Int[],
        line_count=Int[],
        z_slice_count=Int[],
        gap_count=Int[],
        n_reduced_z=Int[],
        n_skipped_frequency=Int[],
        n_kept_frequency=Int[],
        kept_frequency_fraction=Float64[],
        first_skipped_GHz=Union{Missing, Float64}[],
        last_skipped_GHz=Union{Missing, Float64}[],
    )

    for (dataset, outlier) in zip(configurations, outliers)
        skipped = outlier.skipped_frequencies
        push!(
            df,
            (
                dataset.configuration_index,
                dataset.line_count,
                dataset.z_slice_count,
                dataset.gap_count,
                length(dataset.z_abs),
                count(outlier.frequency_skip_mask),
                count(outlier.keep_frequency_mask),
                count(outlier.keep_frequency_mask) / length(ν),
                isempty(skipped) ? missing : first(skipped) * 1e-9,
                isempty(skipped) ? missing : last(skipped) * 1e-9,
            ),
        )
    end
    return df
end

In [ ]:
z_m_FM504_M = 2.298 ± 0.0
rel_err = 0.05
outlier_window = 51
outlier_threshold = 12.0
primary_bf_band = :full
secondary_bf_band = (19e9, 20e9)
top_n_plot = 5

ν, reduced_configurations, full_data, configuration_lookup = load_ET_variation_results(DATA_FILE; z_mirror=z_m_FM504_M.val, rel_err=rel_err)
outliers_by_config = [detect_reduced_ET_outliers(dataset, ν; window=outlier_window, threshold=outlier_threshold) for dataset in reduced_configurations]
ν_clean_by_config = [apply_frequency_mask(ν, outlier.keep_frequency_mask) for outlier in outliers_by_config]
reduced_configurations_clean = [apply_frequency_mask(dataset, outlier.keep_frequency_mask) for (dataset, outlier) in zip(reduced_configurations, outliers_by_config)]
df_outlier_summary = outlier_summary_table(reduced_configurations, outliers_by_config, ν)

println("Loaded $(length(ν)) frequencies and $(length(reduced_configurations)) reduced configurations from $(DATA_FILE).")
println("Full ET: $(length(full_data.z_abs)) z points from $(first(full_data.z_abs)) m to $(last(full_data.z_abs)) m.")
println("Reduced configuration z-count range: $(minimum(df_outlier_summary.n_reduced_z)) to $(maximum(df_outlier_summary.n_reduced_z)).")
println("Skipped reduced frequencies: min=$(minimum(df_outlier_summary.n_skipped_frequency)), median=$(median(df_outlier_summary.n_skipped_frequency)), max=$(maximum(df_outlier_summary.n_skipped_frequency)).")
first(sort(df_outlier_summary, :n_skipped_frequency, rev=true), 10)

## Model Priors

In [ ]:
ϵ_b = 9.23 ± 0.2
Δϵ = ϵ_b - 1
r_b = (2.93e-3 ± 15e-6) / 2
V_b = 4 / 3 * π * r_b^3
α_0 = 3 * Δϵ * V_b / (ϵ_b + 2)

P_in = 1
r_d = 0.15
A = π * r_d^2
σ_Al = (5.0 ± 1) * 1e7

ϵ_d = 9.3 ± 0.1
tanD_d = 1e-5 ± 1e-6
d_d = (1 ± 0.05) * 1e-3

n_disk = 3 ± 0
E0_0 = 1 ± 1
z_m_0 = z_m_FM504_M
σ_m_0 = σ_Al
d_v_i = [8.265 ± 0.1, 9.813 ± 0.1, 9.813 ± 0.1] * 1e-3

p0_all = Dict("E_0"=>E0_0, "z_m"=>z_m_0, "σ_m"=>σ_m_0, "n_disk"=>n_disk)
for i in 1:Int(n_disk.val)
    p0_all["d_v_$i"] = d_v_i[i]
    p0_all["d_d_$i"] = d_d
    p0_all["ϵ_d_$i"] = ϵ_d
    p0_all["tanD_d_$i"] = tanD_d
end
p0_all["r_b"] = r_b
p0_all["ϵ_b"] = ϵ_b

keys_optim = ["E_0", "d_v_1", "d_v_2", "d_v_3"]
keys_helper = ["n_disk"]
keys_fixed = [setdiff(keys(p0_all), keys_optim, keys_helper)...]
keys_all = [keys_optim..., keys_helper..., keys_fixed...]

N_mc = 100
minimum_z_for_optimized_fit = length(keys_optim) + 1

## Shared Analysis Functions

In [ ]:
value_of(x) = hasproperty(x, :val) ? getfield(x, :val) : x
error_of(x) = hasproperty(x, :err) ? getfield(x, :err) : zero(value_of(x))

function final_parameter_table(ν, p_all_ν_mc, p0_all)
    p_final_ν = DataFrame("f"=>ν)
    for key in keys(p0_all)
        p_final_ν[!, key] = mean.(p_all_ν_mc[!, key]) .± std.(p_all_ν_mc[!, key])
    end
    return p_final_ν
end

function integrate_mc_fields(ν, p_all_ν_mc, p0_all, N_mc)
    int_dz_E_mc = zeros(ComplexF64, length(ν), N_mc)
    for f in 1:length(ν), i in 1:N_mc
        p_dict = Dict(key=>p_all_ν_mc[f, key][i] for key in keys(p0_all))
        int_dz_E_mc[f, i] = MADBead.int_dz_E_param(p_dict; f=ν[f])
    end
    return int_dz_E_mc
end

function boostfactor_from_int_dz(ν, int_dz_E_mc; P_in, A, J_0=1)
    ∫dV_E = mean(abs.(int_dz_E_mc), dims=2)[:] .± std(abs.(int_dz_E_mc), dims=2)[:]
    P_sig = J_0^2 / (16 * P_in) .* abs2.(∫dV_E)
    P_0 = MADBead.c_const * A * J_0^2 ./ (2 * MADBead.ϵ0 * (2π .* ν).^2)
    boostfactor = P_sig ./ P_0
    return ∫dV_E, boostfactor
end

function ET_dataframe(ν, dataset)
    df_ET = DataFrame("f"=>ν)
    df_ET[!, "ET"] = [dataset.ET_meas[f, :] for f in 1:length(ν)]
    df_ET[!, "z_abs"] = [dataset.z_abs for _ in 1:length(ν)]
    df_ET[!, "z_rel"] = [dataset.z_rel for _ in 1:length(ν)]
    return df_ET
end

function run_boostfactor_analysis(dataset, ν, p0_all, keys_optim, keys_fixed, keys_helper, N_mc; P_in, A)
    println("Running $(dataset.label) fit with $(length(dataset.z_abs)) z points and $(length(ν)) frequencies.")

    p_all_ν_mc = MADBead.fit_E_z_MC(ν, dataset.z_abs, dataset.ET_meas, p0_all, keys_optim, keys_fixed, keys_helper, N_mc)
    p_final_ν = final_parameter_table(ν, p_all_ν_mc, p0_all)
    int_dz_E_mc = integrate_mc_fields(ν, p_all_ν_mc, p0_all, N_mc)
    ∫dV_E, boostfactor = boostfactor_from_int_dz(ν, int_dz_E_mc; P_in=P_in, A=A)

    df_bf_analysis = DataFrame("f"=>ν, "bf"=>boostfactor, "int_dV_E"=>∫dV_E)
    df_bf_analysis = innerjoin(df_bf_analysis, p_final_ν, on="f")

    df_bf_mc_analysis = DataFrame("f"=>ν)
    df_bf_mc_analysis[!, "int_dV_E_mc"] = [int_dz_E_mc[f, :] for f in 1:length(ν)]
    df_bf_mc_analysis = innerjoin(df_bf_mc_analysis, p_all_ν_mc, on="f")

    return (
        label=dataset.label,
        p_all_ν_mc=p_all_ν_mc,
        p_final_ν=p_final_ν,
        int_dz_E_mc=int_dz_E_mc,
        ∫dV_E=∫dV_E,
        boostfactor=boostfactor,
        df_ET=ET_dataframe(ν, dataset),
        df_bf_analysis=df_bf_analysis,
        df_bf_mc_analysis=df_bf_mc_analysis,
    )
end

function p_nominal_at_frequency(result, row, p0_all)
    return Dict(key=>Float64(value_of(result.p_final_ν[row, key])) for key in keys(p0_all))
end

function fit_quality_metrics(dataset, ν, result, p0_all; rel_floor=0.05)
    residual_sq_sum = 0.0
    reference_sq_sum = 0.0
    chi_sq_sum = 0.0
    rel_abs_errors = Float64[]
    n_points = 0

    for f_idx in eachindex(ν)
        p = p_nominal_at_frequency(result, f_idx, p0_all)
        δ = MADBead.calc_δ_e_mie(p["ϵ_b"], p["r_b"]; f=ν[f_idx])
        δc = MADBead.calc_δ_c_mie(p["ϵ_b"], p["r_b"]; f=ν[f_idx])
        model = sqrt.(abs.(MADBead.E_field2_conv_1D_z_param(dataset.z_abs, p; f=ν[f_idx], δ=δ, δc=δc)))
        reference = abs.(dataset.ET[f_idx, :])
        finite_mask = isfinite.(model) .& isfinite.(reference)
        if !any(finite_mask)
            continue
        end

        model = model[finite_mask]
        reference = reference[finite_mask]
        denom = max.(abs.(reference), eps(Float64))
        residual = model .- reference
        sigma = max.(rel_floor .* denom, eps(Float64))

        residual_sq_sum += sum(abs2, residual)
        reference_sq_sum += sum(abs2, reference)
        chi_sq_sum += sum(abs2, residual ./ sigma)
        append!(rel_abs_errors, abs.(residual) ./ denom)
        n_points += length(reference)
    end

    return (
        fit_abs_nrmse=n_points == 0 ? NaN : sqrt(residual_sq_sum / max(reference_sq_sum, eps(Float64))),
        fit_rel_abs_median=isempty(rel_abs_errors) ? NaN : median(rel_abs_errors),
        fit_rel_abs_p90=isempty(rel_abs_errors) ? NaN : quantile(rel_abs_errors, 0.9),
        fit_chi_rms=n_points == 0 ? NaN : sqrt(chi_sq_sum / n_points),
        fit_n_points=n_points,
    )
end

function reference_indices(ν_query, ν_reference)
    indices = Vector{Int}(undef, length(ν_query))
    for (i, f) in enumerate(ν_query)
        j = searchsortedfirst(ν_reference, f)
        if j > length(ν_reference) || ν_reference[j] != f
            j = argmin(abs.(ν_reference .- f))
            @assert abs(ν_reference[j] - f) <= max(abs(f), 1.0) * 1e-12 "Could not match frequency $(f) to reference grid."
        end
        indices[i] = j
    end
    return indices
end

function boostfactor_agreement_metrics(ν_candidate, bf_candidate, ν_reference, bf_reference; band=(minimum(ν_reference), maximum(ν_reference)))
    band_mask = (ν_candidate .>= band[1]) .& (ν_candidate .<= band[2])
    if !any(band_mask)
        return (bf_nrmse=NaN, bf_rel_abs_median=NaN, bf_rel_abs_p90=NaN, bf_corr=NaN, bf_peak_delta_GHz=NaN, bf_n_points=0)
    end

    ν_band = ν_candidate[band_mask]
    candidate = Float64.(value_of.(bf_candidate[band_mask]))
    reference = Float64.(value_of.(bf_reference[reference_indices(ν_band, ν_reference)]))
    finite_mask = isfinite.(candidate) .& isfinite.(reference)
    if !any(finite_mask)
        return (bf_nrmse=NaN, bf_rel_abs_median=NaN, bf_rel_abs_p90=NaN, bf_corr=NaN, bf_peak_delta_GHz=NaN, bf_n_points=0)
    end

    ν_band = ν_band[finite_mask]
    candidate = candidate[finite_mask]
    reference = reference[finite_mask]
    residual = candidate .- reference
    rel_abs = abs.(residual) ./ max.(abs.(reference), eps(Float64))
    peak_delta_GHz = (ν_band[argmax(candidate)] - ν_band[argmax(reference)]) * 1e-9

    return (
        bf_nrmse=sqrt(sum(abs2, residual) / max(sum(abs2, reference), eps(Float64))),
        bf_rel_abs_median=median(rel_abs),
        bf_rel_abs_p90=quantile(rel_abs, 0.9),
        bf_corr=length(candidate) > 1 ? cor(candidate, reference) : NaN,
        bf_peak_delta_GHz=peak_delta_GHz,
        bf_n_points=length(candidate),
    )
end

function configuration_metrics_dataframe(configurations, clean_configurations, ν_clean_by_config, results, outliers, full_result, ν_full, p0_all, keys_optim; secondary_band=(19e9, 20e9))
    rows = NamedTuple[]
    full_bf = full_result.boostfactor
    full_band = (minimum(ν_full), maximum(ν_full))

    for (dataset, clean_dataset, ν_clean, result, outlier) in zip(configurations, clean_configurations, ν_clean_by_config, results, outliers)
        fit_metrics = fit_quality_metrics(clean_dataset, ν_clean, result, p0_all; rel_floor=rel_err)
        bf_full = boostfactor_agreement_metrics(ν_clean, result.boostfactor, ν_full, full_bf; band=full_band)
        bf_secondary = boostfactor_agreement_metrics(ν_clean, result.boostfactor, ν_full, full_bf; band=secondary_band)

        push!(
            rows,
            merge(
                (
                    configuration_index=dataset.configuration_index,
                    line_count=dataset.line_count,
                    z_slice_count=dataset.z_slice_count,
                    gap_count=dataset.gap_count,
                    n_reduced_z=length(dataset.z_abs),
                    n_kept_frequency=length(ν_clean),
                    n_skipped_frequency=count(outlier.frequency_skip_mask),
                    kept_frequency_fraction=length(ν_clean) / length(ν_full),
                    fit_has_enough_z=length(dataset.z_abs) >= length(keys_optim) + 1,
                ),
                fit_metrics,
                (
                    bf_nrmse_fullband=bf_full.bf_nrmse,
                    bf_rel_abs_median_fullband=bf_full.bf_rel_abs_median,
                    bf_rel_abs_p90_fullband=bf_full.bf_rel_abs_p90,
                    bf_corr_fullband=bf_full.bf_corr,
                    bf_peak_delta_GHz_fullband=bf_full.bf_peak_delta_GHz,
                    bf_n_points_fullband=bf_full.bf_n_points,
                    bf_nrmse_19_20GHz=bf_secondary.bf_nrmse,
                    bf_rel_abs_median_19_20GHz=bf_secondary.bf_rel_abs_median,
                    bf_rel_abs_p90_19_20GHz=bf_secondary.bf_rel_abs_p90,
                    bf_corr_19_20GHz=bf_secondary.bf_corr,
                    bf_peak_delta_GHz_19_20GHz=bf_secondary.bf_peak_delta_GHz,
                    bf_n_points_19_20GHz=bf_secondary.bf_n_points,
                ),
            ),
        )
    end

    df = DataFrame(rows)
    df[!, :rank_bf_fullband] = invperm(sortperm(df.bf_nrmse_fullband))
    df[!, :rank_fit_abs_nrmse] = invperm(sortperm(df.fit_abs_nrmse))
    return df
end

## ET And Outlier Diagnostics

In [ ]:
function plot_ET_magnitude(ν, dataset; title=dataset.label)
    color_fraction = length(dataset.z_abs) == 1 ? [0.5] : (dataset.z_abs .- first(dataset.z_abs)) ./ (last(dataset.z_abs) - first(dataset.z_abs))
    colors = cgrad([:blue, :red])[color_fraction]
    plt = plot(xlabel=L"ν \ \mathrm{[GHz]}", ylabel=L"|\int dA E|", title=title)
    for i in eachindex(dataset.z_abs)
        plot!(
            plt,
            ν .* 1e-9,
            getfield.(abs.(dataset.ET_meas[:, i]), :val),
            ribbon=getfield.(abs.(dataset.ET_meas[:, i]), :err),
            label="",
            c=colors[i],
            lw=1,
        )
    end
    return plt
end

function plot_reduced_outlier_components(ν, dataset, outliers; title_prefix=dataset.label)
    color_fraction = length(dataset.z_abs) == 1 ? [0.5] : (dataset.z_abs .- first(dataset.z_abs)) ./ (last(dataset.z_abs) - first(dataset.z_abs))
    colors = cgrad([:blue, :red])[color_fraction]
    p_re = plot(xlabel=L"ν \ \mathrm{[GHz]}", ylabel=L"\mathrm{Re}(\int dA E)", title="$(title_prefix) Re(ET)")
    p_im = plot(xlabel=L"ν \ \mathrm{[GHz]}", ylabel=L"\mathrm{Im}(\int dA E)", title="$(title_prefix) Im(ET)")

    for i in eachindex(dataset.z_abs)
        z_label = "z=$(round(dataset.z_rel[i] * 1e3, digits=1)) mm rel"
        plot!(p_re, ν .* 1e-9, real.(dataset.ET[:, i]), c=colors[i], lw=1, alpha=0.75, label=z_label)
        plot!(p_im, ν .* 1e-9, imag.(dataset.ET[:, i]), c=colors[i], lw=1, alpha=0.75, label=z_label)

        flagged = outliers.point_mask[:, i]
        if any(flagged)
            scatter!(p_re, ν[flagged] .* 1e-9, real.(dataset.ET[flagged, i]), c=:black, ms=3, label="")
            scatter!(p_im, ν[flagged] .* 1e-9, imag.(dataset.ET[flagged, i]), c=:black, ms=3, label="")
        end
    end

    return plot(p_re, p_im, layout=(2, 1), size=(1000, 650), margin=5mm)
end

function plot_outlier_skip_summary(df)
    sorted_df = sort(df, [:line_count, :z_slice_count, :gap_count])
    labels = ["$(r.line_count)-$(r.z_slice_count)-$(r.gap_count)" for r in eachrow(sorted_df)]
    plt = bar(
        string.(sorted_df.configuration_index),
        sorted_df.n_skipped_frequency,
        group=sorted_df.gap_count,
        xlabel="configuration index",
        ylabel="skipped frequencies",
        title="Reduced ET cleanup by configuration",
        legendtitle="gap count",
        lw=0,
        size=(1100, 420),
        margin=5mm,
    )
    return plt
end

max_z_config_pos = argmax([length(dataset.z_abs) for dataset in reduced_configurations])
save_notebook_plot(
    plot(
        plot_ET_magnitude(ν, reduced_configurations[max_z_config_pos]; title="largest reduced configuration"),
        plot_ET_magnitude(ν, full_data; title="full"),
        layout=(1, 2),
        size=(1050, 400),
        margin=5mm,
    ),
    "01_et_magnitude_overview",
)

In [ ]:
worst_outlier_pos = argmax(df_outlier_summary.n_skipped_frequency)
println("Most skipped configuration: $(reduced_configurations[worst_outlier_pos].label) with $(df_outlier_summary.n_skipped_frequency[worst_outlier_pos]) skipped frequencies.")
save_notebook_plot(
    plot(
        plot_outlier_skip_summary(df_outlier_summary),
        plot_reduced_outlier_components(ν, reduced_configurations[worst_outlier_pos], outliers_by_config[worst_outlier_pos]),
        layout=(2, 1),
        size=(1100, 1050),
        margin=6mm,
    ),
    "02_reduced_outlier_diagnostics",
)

## Run Full And Configuration Fits

In [ ]:
USE_CACHED_PRODUCTION_RESULTS = true

if USE_CACHED_PRODUCTION_RESULTS
    @assert isfile(RESULTS_FILE) "Missing cached production results: $(RESULTS_FILE). Run scripts/slurm/submit_spider_et_variation_production.sh first."
    cached_results = JLD2.load(RESULTS_FILE)
    for (name, value) in cached_results
        if name in ["DATA_FILE"]
            continue
        end
        Core.eval(@__MODULE__, Expr(:(=), Symbol(name), QuoteNode(value)))
    end
    loaded_from_cache = true
    println("Loaded cached production results from $(RESULTS_FILE).")
    println("Best boostfactor reconstruction: $(reduced_configurations[best_bf_pos].label), NRMSE=$(df_config_metrics_by_bf.bf_nrmse_fullband[1]).")
    println("Best MADBead fit residual: $(reduced_configurations[best_fit_pos].label), NRMSE=$(df_config_metrics_by_fit.fit_abs_nrmse[1]).")
    first(df_config_metrics_by_bf, 10)
else
    loaded_from_cache = false
    full_result = run_boostfactor_analysis(full_data, ν, p0_all, keys_optim, keys_fixed, keys_helper, N_mc; P_in=P_in, A=A);

    configuration_results = Vector{Any}(undef, length(reduced_configurations_clean))
    for i in eachindex(reduced_configurations_clean)
        println("Configuration $(i)/$(length(reduced_configurations_clean)): $(reduced_configurations[i].label)")
        configuration_results[i] = run_boostfactor_analysis(reduced_configurations_clean[i], ν_clean_by_config[i], p0_all, keys_optim, keys_fixed, keys_helper, N_mc; P_in=P_in, A=A)
    end

    @assert length(full_result.df_bf_analysis.f) == length(ν) "Full fit should use the complete frequency grid."
    for i in eachindex(configuration_results)
        @assert isempty(intersect(configuration_results[i].df_bf_analysis.f, outliers_by_config[i].skipped_frequencies)) "Skipped frequencies leaked into $(reduced_configurations[i].label) fit output."
    end

    df_config_metrics = configuration_metrics_dataframe(
        reduced_configurations,
        reduced_configurations_clean,
        ν_clean_by_config,
        configuration_results,
        outliers_by_config,
        full_result,
        ν,
        p0_all,
        keys_optim;
        secondary_band=secondary_bf_band,
    )

    df_config_metrics_by_bf = sort(df_config_metrics, :bf_nrmse_fullband)
    df_config_metrics_by_fit = sort(df_config_metrics, :fit_abs_nrmse)
    config_position_by_index = Dict(dataset.configuration_index => i for (i, dataset) in enumerate(reduced_configurations))
    best_bf_pos = config_position_by_index[df_config_metrics_by_bf.configuration_index[1]]
    best_fit_pos = config_position_by_index[df_config_metrics_by_fit.configuration_index[1]]

    println("Best boostfactor reconstruction: $(reduced_configurations[best_bf_pos].label), NRMSE=$(df_config_metrics_by_bf.bf_nrmse_fullband[1]).")
    println("Best MADBead fit residual: $(reduced_configurations[best_fit_pos].label), NRMSE=$(df_config_metrics_by_fit.fit_abs_nrmse[1]).")
    first(df_config_metrics_by_bf, 10)
end


## Configuration Overview

In [ ]:
function config_display_labels(df)
    return ["cfg$(r.configuration_index) L$(r.line_count) Z$(r.z_slice_count) G$(r.gap_count)" for r in eachrow(df)]
end

function plot_top_metric_bars(df, metric; n=10, title="")
    top = first(sort(df, metric), min(n, nrow(df)))
    labels = config_display_labels(top)
    return bar(
        reverse(labels),
        reverse(top[!, metric]),
        orientation=:horizontal,
        xlabel=String(metric),
        title=title,
        legend=false,
        size=(820, 520),
        margin=7mm,
    )
end

function metric_matrix(df, metric, gap_count)
    lines = sort(unique(df.line_count))
    z_slices = sort(unique(df.z_slice_count))
    mat = fill(NaN, length(lines), length(z_slices))
    for (i, line_count) in enumerate(lines), (j, z_slice_count) in enumerate(z_slices)
        rows = df[(df.line_count .== line_count) .& (df.z_slice_count .== z_slice_count) .& (df.gap_count .== gap_count), :]
        if nrow(rows) == 1
            mat[i, j] = rows[1, metric]
        end
    end
    return lines, z_slices, mat
end

function plot_metric_heatmaps(df, metric; title="")
    gaps = sort(unique(df.gap_count))
    plts = Any[]
    for gap in gaps
        lines, z_slices, mat = metric_matrix(df, metric, gap)
        z_ticks = (z_slices, string.(z_slices))
        line_ticks = (lines, string.(lines))
        push!(
            plts,
            heatmap(
                z_slices,
                lines,
                mat,
                xlabel="z-slice count",
                ylabel="line count",
                title="$(title), gaps=$(gap)",
                colorbar_title=String(metric),
                xticks=z_ticks,
                yticks=line_ticks,
                xlims=(minimum(z_slices) - 0.5, maximum(z_slices) + 0.5),
                ylims=(minimum(lines) - 0.5, maximum(lines) + 0.5),
                yflip=false,
                titlefontsize=16,
                guidefontsize=13,
                tickfontsize=11,
                left_margin=14mm,
                right_margin=12mm,
                bottom_margin=10mm,
                top_margin=8mm,
            ),
        )
    end
    return plot(
        plts...,
        layout=(2, 2),
        size=(1300, 1050),
        left_margin=16mm,
        right_margin=12mm,
        bottom_margin=12mm,
        top_margin=10mm,
    )
end

function plot_fit_vs_bf_score(df)
    return scatter(
        df.fit_abs_nrmse,
        df.bf_nrmse_fullband,
        group=df.gap_count,
        xlabel="MADBead fit magnitude NRMSE",
        ylabel="boostfactor NRMSE vs full",
        title="Fit quality vs boostfactor reconstruction",
        legendtitle="gap count",
        ms=6,
        size=(760, 560),
        margin=5mm,
    )
end

save_notebook_plot(
    plot(
        plot_top_metric_bars(df_config_metrics, :bf_nrmse_fullband; n=10, title="Best boostfactor reconstruction"),
        plot_top_metric_bars(df_config_metrics, :fit_abs_nrmse; n=10, title="Best MADBead fit residual"),
        plot_fit_vs_bf_score(df_config_metrics),
        layout=(1, 3),
        size=(1700, 560),
        margin=6mm,
    ),
    "03_configuration_ranking_overview",
)

In [ ]:
save_notebook_plot(
    plot(
        plot_metric_heatmaps(df_config_metrics, :bf_nrmse_fullband; title="boostfactor NRMSE vs full"),
        plot_metric_heatmaps(df_config_metrics, :fit_abs_nrmse; title="MADBead fit NRMSE"),
        layout=(1, 2),
        size=(2700, 1250),
        left_margin=18mm,
        right_margin=12mm,
        bottom_margin=12mm,
        top_margin=12mm,
    ),
    "04_configuration_metric_heatmaps",
)

## Representative Fit Diagnostics

In [ ]:
function plot_representative_fit(ν, dataset, result, p0_all; mc_idx=1, title_prefix=dataset.label, frequency=nothing)
    freq_idx = frequency === nothing ? argmax(abs.(value_of.(result.p_final_ν[!, "E_0"]))) : findmin(abs.(ν .- frequency))[2]
    p_select = Dict(key=>result.p_all_ν_mc[freq_idx, key][mc_idx] for key in keys(p0_all))

    distance_fit, ignore, ignore = MADBead.fit_param_to_model(p_select)
    z_σ = sum(distance_fit) .+ [0, 1e-3]
    z_ϵ = [sum(distance_fit[2*i+1:end]) .+ [0, -1e-3] for i in 1:Int(p0_all["n_disk"].val)]
    z_pos_fit = (first(dataset.z_abs) - 0.04):1e-4:(last(dataset.z_abs) + 0.01)

    δ = MADBead.calc_δ_e_mie(p_select["ϵ_b"], p_select["r_b"]; f=ν[freq_idx])
    δc = MADBead.calc_δ_c_mie(p_select["ϵ_b"], p_select["r_b"]; f=ν[freq_idx])
    fit_plot = sqrt.(abs.(MADBead.E_field2_conv_1D_z_param(z_pos_fit, p_select; f=ν[freq_idx], δ=δ, δc=δc)))
    deconv_plot = abs.(MADBead.E_field_1D_z_param(z_pos_fit, p_select; f=ν[freq_idx]))

    plt = plot(title="$(title_prefix), $(round(ν[freq_idx] * 1e-9, digits=3)) GHz")
    vspan!(plt, z_σ .* 1e3, label="", c=:darkgrey, alpha=0.35)
    for region in z_ϵ
        vspan!(plt, region .* 1e3, label="", c=:lightgrey, alpha=0.45)
    end
    scatter!(plt, dataset.z_abs .* 1e3, abs.(dataset.ET_meas[freq_idx, :]), msw=1, c=2, label="ET")
    plot!(plt, z_pos_fit .* 1e3, fit_plot, c=4, ls=:dash, label="Fit")
    plot!(plt, z_pos_fit .* 1e3, deconv_plot, c=3, ls=:dashdot, label="Deconvoluted")
    plot!(plt, xlabel=L"z \: \mathrm{[mm]}", ylabel=L"E_{1D} \: \mathrm{[Vm]}")
    return plt
end

save_notebook_plot(
    plot(
        plot_representative_fit(ν_clean_by_config[best_bf_pos], reduced_configurations_clean[best_bf_pos], configuration_results[best_bf_pos], p0_all; title_prefix="best BF $(reduced_configurations[best_bf_pos].label)"),
        plot_representative_fit(ν_clean_by_config[best_fit_pos], reduced_configurations_clean[best_fit_pos], configuration_results[best_fit_pos], p0_all; title_prefix="best fit $(reduced_configurations[best_fit_pos].label)"),
        plot_representative_fit(ν, full_data, full_result, p0_all; title_prefix="full reference"),
        layout=(1, 3),
        size=(1500, 470),
        margin=6mm,
    ),
    "05_representative_fit_diagnostics",
)

## Fitted Parameters

In [ ]:
function plot_fit_parameters(ν, result, keys_optim; title=result.label)
    ncol = ceil(Int, sqrt(length(keys_optim)))
    nrow = ceil(Int, length(keys_optim) / ncol)
    plt = plot(layout=(nrow, ncol), size=(900, 650))
    for (sp, key) in enumerate(keys_optim)
        plot!(
            plt,
            sp=sp,
            ν .* 1e-9,
            value_of.(result.p_final_ν[!, key]),
            ribbon=error_of.(result.p_final_ν[!, key]),
            label="",
            c=2,
            xlabel=L"ν \: \mathrm{[GHz]}",
            ylabel=key,
            title=title,
        )
    end
    return plt
end

save_notebook_plot(
    plot(
        plot_fit_parameters(ν_clean_by_config[best_bf_pos], configuration_results[best_bf_pos], keys_optim; title="best BF $(reduced_configurations[best_bf_pos].label)"),
        plot_fit_parameters(ν, full_result, keys_optim; title="full reference"),
        layout=(1, 2),
        size=(1200, 650),
        margin=5mm,
    ),
    "06_fit_parameter_traces",
)

## Boostfactor Reconstruction

In [ ]:
function plot_boostfactor_overlay(ν_full, full_result, configurations, ν_clean_by_config, results, ranked_metrics; n=5)
    plt = plot(
        ν_full .* 1e-9,
        value_of.(full_result.boostfactor),
        ribbon=error_of.(full_result.boostfactor),
        lw=2,
        c=:black,
        label="full reference",
        xlabel=L"ν \ \mathrm{[GHz]}",
        ylabel=L"\beta^2",
        title="Top boostfactor reconstructions",
        size=(1000, 560),
        margin=5mm,
    )

    top = first(ranked_metrics, min(n, nrow(ranked_metrics)))
    for row in eachrow(top)
        pos = config_position_by_index[row.configuration_index]
        plot!(
            plt,
            ν_clean_by_config[pos] .* 1e-9,
            value_of.(results[pos].boostfactor),
            lw=1.25,
            label="cfg$(row.configuration_index) L$(row.line_count) Z$(row.z_slice_count) G$(row.gap_count)",
        )
    end
    return plt
end

save_notebook_plot(
    plot_boostfactor_overlay(ν, full_result, reduced_configurations, ν_clean_by_config, configuration_results, df_config_metrics_by_bf; n=top_n_plot),
    "07_top_boostfactor_reconstructions",
)

function metric_row_for_config(df, configuration_index)
    rows = df[df.configuration_index .== configuration_index, :]
    @assert nrow(rows) == 1 "Expected exactly one metrics row for configuration $(configuration_index)."
    return rows[1, :]
end

function plot_configuration_boostfactor(ν_full, full_result, ν_config, config_result, config, metrics_row; plot_kwargs...)
    label = "cfg$(config.configuration_index) L$(config.line_count) Z$(config.z_slice_count) G$(config.gap_count)"
    nrmse = round(metrics_row.bf_nrmse_fullband, digits=4)
    plt = plot(
        ν_full .* 1e-9,
        value_of.(full_result.boostfactor),
        ribbon=error_of.(full_result.boostfactor),
        lw=2,
        c=:black,
        label="full reference",
        xlabel=L"ν \ \mathrm{[GHz]}",
        ylabel=L"\beta^2",
        title="$(label) boostfactor vs full, NRMSE=$(nrmse)",
        size=(1000, 560),
        margin=6mm,
    )
    plot!(
        plt,
        ν_config .* 1e-9,
        value_of.(config_result.boostfactor),
        ribbon=error_of.(config_result.boostfactor),
        lw=1.5,
        c=2,
        label="reduced configuration",
    )
    plot!(plt; plot_kwargs...)
    return plt
end

function save_all_configuration_fit_and_boostfactor_plots()
    saved_paths = String[]
    for pos in eachindex(reduced_configurations_clean)
        config = reduced_configurations[pos]
        clean_config = reduced_configurations_clean[pos]
        result = configuration_results[pos]
        ν_config = ν_clean_by_config[pos]
        metrics_row = metric_row_for_config(df_config_metrics, config.configuration_index)

        fit_plot = plot_representative_fit(ν_config, clean_config, result, p0_all; title_prefix=config.label)
        plot!(fit_plot, size=(900, 560), margin=6mm)
        append!(saved_paths, save_config_plot(fit_plot, config, "boostfactor_fit_efield_vs_z"))

        bf_plot = plot_configuration_boostfactor(ν, full_result, ν_config, result, config, metrics_row)
        append!(saved_paths, save_config_plot(bf_plot, config, "boostfactor_vs_full"))
    end
    println("Saved $(length(saved_paths)) per-configuration boostfactor analysis plot files under $(joinpath(SPIDER_REPO, "figs", "configs")).")
    return saved_paths
end

configuration_plot_files = save_all_configuration_fit_and_boostfactor_plots();
first(configuration_plot_files, min(8, length(configuration_plot_files)))

## Focused Full Vs L3 Z1 G4 Comparison

This section compares the full reference only to the configuration with 4 gaps, 3 lines, and 1 z-slice per gap.

In [ ]:
focused_comparison_fig_dir = joinpath(SPIDER_REPO, "figs", "comparisons", "full_vs_lines03_zslices01_gaps04")
mkpath(focused_comparison_fig_dir)

function save_focused_comparison_plot(plt, stem; formats=("png", "svg"))
    for ext in formats
        savefig(plt, joinpath(focused_comparison_fig_dir, "$(stem).$(ext)"))
    end
    return plt
end

function focused_config_selection()
    rows = df_config_metrics[(df_config_metrics.line_count .== 3) .& (df_config_metrics.z_slice_count .== 1) .& (df_config_metrics.gap_count .== 4), :]
    @assert nrow(rows) == 1 "Expected exactly one focused comparison configuration."
    config_index = rows.configuration_index[1]
    pos = config_position_by_index[config_index]
    return rows[1, :], pos, reduced_configurations[pos], reduced_configurations_clean[pos], configuration_results[pos], ν_clean_by_config[pos]
end

function plot_boostfactor_fractional_residual(ν_full, full_result, ν_config, config_result, config)
    frequency_to_index = Dict(f => i for (i, f) in enumerate(ν_full))
    full_idx = [frequency_to_index[f] for f in ν_config]
    full_bf = value_of.(full_result.boostfactor[full_idx])
    config_bf = value_of.(config_result.boostfactor)
    rel_residual = (config_bf .- full_bf) ./ max.(abs.(full_bf), eps(Float64))
    label = "cfg$(config.configuration_index) L$(config.line_count) Z$(config.z_slice_count) G$(config.gap_count)"

    plt = plot(
        ν_config .* 1e-9,
        rel_residual,
        xlabel="frequency [GHz]",
        ylabel="(config - full) / full",
        title="$(label) boostfactor fractional residual",
        label="",
        lw=1.5,
        c=2,
        size=(1000, 420),
        margin=6mm,
    )
    hline!(plt, [0.0], c=:black, ls=:dash, lw=1, label="")
    return plt
end

focused_metrics_row, focused_config_pos, focused_config, focused_config_clean, focused_result, ν_focused = focused_config_selection()
focused_label = "cfg$(focused_config.configuration_index) L$(focused_config.line_count) Z$(focused_config.z_slice_count) G$(focused_config.gap_count)"
focused_fit_frequency = 19.5e9
@assert minimum(abs.(ν .- focused_fit_frequency)) == 0.0 "Full reference does not contain 19.5 GHz exactly."
@assert minimum(abs.(ν_focused .- focused_fit_frequency)) == 0.0 "$(focused_label) cleaned frequency grid does not contain 19.5 GHz exactly."

println("Focused comparison: $(focused_label)")
println("Focused fitted-field comparison frequency: $(focused_fit_frequency * 1e-9) GHz")
println("Full-band boostfactor NRMSE: $(focused_metrics_row.bf_nrmse_fullband)")
println("Saved focused comparison plots to $(focused_comparison_fig_dir)")

focused_et_plot = save_focused_comparison_plot(
    plot(
        plot_ET_magnitude(ν_focused, focused_config_clean; title="$(focused_label) cleaned ET"),
        plot_ET_magnitude(ν, full_data; title="full ET"),
        layout=(1, 2),
        size=(1200, 440),
        margin=6mm,
    ),
    "01_et_magnitude_full_vs_config",
)

focused_fit_plot = save_focused_comparison_plot(
    plot(
        plot_representative_fit(ν_focused, focused_config_clean, focused_result, p0_all; title_prefix=focused_label, frequency=focused_fit_frequency),
        plot_representative_fit(ν, full_data, full_result, p0_all; title_prefix="full reference", frequency=focused_fit_frequency),
        layout=(1, 2),
        size=(1400, 520),
        margin=6mm,
    ),
    "02_fit_efield_vs_z_full_vs_config",
)

focused_params_plot = save_focused_comparison_plot(
    plot(
        plot_fit_parameters(ν_focused, focused_result, keys_optim; title=focused_label),
        plot_fit_parameters(ν, full_result, keys_optim; title="full reference"),
        layout=(1, 2),
        size=(1300, 700),
        margin=6mm,
    ),
    "03_fit_parameters_full_vs_config",
)

focused_boostfactor_plot = save_focused_comparison_plot(
    plot_configuration_boostfactor(ν, full_result, ν_focused, focused_result, focused_config, focused_metrics_row),
    "04_boostfactor_full_vs_config",
)

focused_residual_plot = save_focused_comparison_plot(
    plot_boostfactor_fractional_residual(ν, full_result, ν_focused, focused_result, focused_config),
    "05_boostfactor_fractional_residual",
)

focused_comparison_overview = plot(
    focused_et_plot,
    focused_fit_plot,
    focused_params_plot,
    focused_boostfactor_plot,
    focused_residual_plot,
    layout=(5, 1),
    size=(1500, 3400),
    margin=8mm,
)
save_focused_comparison_plot(focused_comparison_overview, "00_full_vs_config_overview")

## Focused Full Vs L3 Z2 G4 Comparison

This section compares the full reference only to the configuration with 4 gaps, 3 lines, and 2 z-slices per gap.

In [ ]:
l3z2g4_comparison_fig_dir = joinpath(SPIDER_REPO, "figs", "comparisons", "full_vs_lines03_zslices02_gaps04")
mkpath(l3z2g4_comparison_fig_dir)

function save_l3z2g4_comparison_plot(plt, stem; formats=("png", "svg"))
    for ext in formats
        savefig(plt, joinpath(l3z2g4_comparison_fig_dir, "$(stem).$(ext)"))
    end
    return plt
end

function l3z2g4_config_selection()
    rows = df_config_metrics[(df_config_metrics.line_count .== 3) .& (df_config_metrics.z_slice_count .== 2) .& (df_config_metrics.gap_count .== 4), :]
    @assert nrow(rows) == 1 "Expected exactly one L3 Z2 G4 comparison configuration."
    config_index = rows.configuration_index[1]
    pos = config_position_by_index[config_index]
    return rows[1, :], pos, reduced_configurations[pos], reduced_configurations_clean[pos], configuration_results[pos], ν_clean_by_config[pos]
end

l3z2g4_metrics_row, l3z2g4_config_pos, l3z2g4_config, l3z2g4_config_clean, l3z2g4_result, ν_l3z2g4 = l3z2g4_config_selection()
l3z2g4_label = "cfg$(l3z2g4_config.configuration_index) L$(l3z2g4_config.line_count) Z$(l3z2g4_config.z_slice_count) G$(l3z2g4_config.gap_count)"

println("Focused comparison: $(l3z2g4_label)")
println("Full-band boostfactor NRMSE: $(l3z2g4_metrics_row.bf_nrmse_fullband)")
println("Saved focused comparison plots to $(l3z2g4_comparison_fig_dir)")

l3z2g4_et_plot = save_l3z2g4_comparison_plot(
    plot(
        plot_ET_magnitude(ν_l3z2g4, l3z2g4_config_clean; title="$(l3z2g4_label) cleaned ET"),
        plot_ET_magnitude(ν, full_data; title="full ET"),
        layout=(1, 2),
        size=(1200, 440),
        margin=6mm,
    ),
    "01_et_magnitude_full_vs_config",
)

l3z2g4_fit_plot = save_l3z2g4_comparison_plot(
    plot(
        plot_representative_fit(ν_l3z2g4, l3z2g4_config_clean, l3z2g4_result, p0_all; title_prefix=l3z2g4_label),
        plot_representative_fit(ν, full_data, full_result, p0_all; title_prefix="full reference"),
        layout=(1, 2),
        size=(1400, 520),
        margin=6mm,
    ),
    "02_fit_efield_vs_z_full_vs_config",
)

l3z2g4_params_plot = save_l3z2g4_comparison_plot(
    plot(
        plot_fit_parameters(ν_l3z2g4, l3z2g4_result, keys_optim; title=l3z2g4_label),
        plot_fit_parameters(ν, full_result, keys_optim; title="full reference"),
        layout=(1, 2),
        size=(1300, 700),
        margin=6mm,
    ),
    "03_fit_parameters_full_vs_config",
)

l3z2g4_boostfactor_plot = save_l3z2g4_comparison_plot(
    plot_configuration_boostfactor(ν, full_result, ν_l3z2g4, l3z2g4_result, l3z2g4_config, l3z2g4_metrics_row; xlims=(19, 20)),
    "04_boostfactor_full_vs_config",
)

l3z2g4_residual_plot = save_l3z2g4_comparison_plot(
    plot_boostfactor_fractional_residual(ν, full_result, ν_l3z2g4, l3z2g4_result, l3z2g4_config),
    "05_boostfactor_fractional_residual",
)

l3z2g4_comparison_overview = plot(
    l3z2g4_et_plot,
    l3z2g4_fit_plot,
    l3z2g4_params_plot,
    l3z2g4_boostfactor_plot,
    l3z2g4_residual_plot,
    layout=(5, 1),
    size=(1500, 3400),
    margin=8mm,
)
save_l3z2g4_comparison_plot(l3z2g4_comparison_overview, "00_full_vs_config_overview")

## Save Results

In [ ]:
if @isdefined(loaded_from_cache) && loaded_from_cache
    println("Loaded cached production results from $(RESULTS_FILE); skipping notebook-side save.")
else
    mkpath(dirname(RESULTS_FILE))

    ν_full = ν
    ν_by_config = ν_clean_by_config
    configuration_metadata = select(df_config_metrics, :configuration_index, :line_count, :z_slice_count, :gap_count, :n_reduced_z)

    outlier_point_mask_by_config = [outlier.point_mask for outlier in outliers_by_config]
    outlier_real_score_by_config = [outlier.real_score for outlier in outliers_by_config]
    outlier_imag_score_by_config = [outlier.imag_score for outlier in outliers_by_config]
    outlier_frequency_skip_mask_by_config = [outlier.frequency_skip_mask for outlier in outliers_by_config]
    outlier_keep_frequency_mask_by_config = [outlier.keep_frequency_mask for outlier in outliers_by_config]
    skipped_frequencies_by_config = [outlier.skipped_frequencies for outlier in outliers_by_config]

    z_reduced_abs_by_config = [dataset.z_abs for dataset in reduced_configurations]
    z_reduced_rel_by_config = [dataset.z_rel for dataset in reduced_configurations]
    z_ixs_used_by_config = [dataset.z_ixs_used for dataset in reduced_configurations]
    ET_reduced_raw_by_config = [dataset.ET for dataset in reduced_configurations]
    ET_reduced_meas_raw_by_config = [dataset.ET_meas for dataset in reduced_configurations]
    ET_reduced_clean_by_config = [dataset.ET for dataset in reduced_configurations_clean]
    ET_reduced_meas_clean_by_config = [dataset.ET_meas for dataset in reduced_configurations_clean]

    p_all_ν_mc_by_config = [result.p_all_ν_mc for result in configuration_results]
    p_final_ν_by_config = [result.p_final_ν for result in configuration_results]
    int_dz_E_mc_by_config = [result.int_dz_E_mc for result in configuration_results]
    ∫dV_E_by_config = [result.∫dV_E for result in configuration_results]
    boostfactor_by_config = [result.boostfactor for result in configuration_results]
    df_ET_by_config = [result.df_ET for result in configuration_results]
    df_ET_raw_by_config = [ET_dataframe(ν, dataset) for dataset in reduced_configurations]
    df_bf_analysis_by_config = [result.df_bf_analysis for result in configuration_results]
    df_bf_mc_analysis_by_config = [result.df_bf_mc_analysis for result in configuration_results]

    z_full_abs = full_data.z_abs
    z_full_rel = full_data.z_rel
    ET_full = full_data.ET
    ET_full_meas = full_data.ET_meas
    p_all_ν_mc_full = full_result.p_all_ν_mc
    p_final_ν_full = full_result.p_final_ν
    int_dz_E_mc_full = full_result.int_dz_E_mc
    ∫dV_E_full = full_result.∫dV_E
    boostfactor_full = full_result.boostfactor
    df_ET_full = full_result.df_ET
    df_bf_analysis_full = full_result.df_bf_analysis
    df_bf_mc_analysis_full = full_result.df_bf_mc_analysis

    JLD2.jldsave(
        RESULTS_FILE;
        DATA_FILE,
        ν,
        ν_full,
        ν_by_config,
        z_m_FM504_M,
        rel_err,
        N_mc,
        p0_all,
        keys_optim,
        keys_helper,
        keys_fixed,
        primary_bf_band,
        secondary_bf_band,
        configuration_lookup,
        configuration_metadata,
        df_outlier_summary,
        df_config_metrics,
        df_config_metrics_by_bf,
        df_config_metrics_by_fit,
        outlier_point_mask_by_config,
        outlier_real_score_by_config,
        outlier_imag_score_by_config,
        outlier_frequency_skip_mask_by_config,
        outlier_keep_frequency_mask_by_config,
        skipped_frequencies_by_config,
        outlier_window,
        outlier_threshold,
        z_reduced_abs_by_config,
        z_reduced_rel_by_config,
        z_ixs_used_by_config,
        ET_reduced_raw_by_config,
        ET_reduced_meas_raw_by_config,
        ET_reduced_clean_by_config,
        ET_reduced_meas_clean_by_config,
        p_all_ν_mc_by_config,
        p_final_ν_by_config,
        int_dz_E_mc_by_config,
        ∫dV_E_by_config,
        boostfactor_by_config,
        df_ET_by_config,
        df_ET_raw_by_config,
        df_bf_analysis_by_config,
        df_bf_mc_analysis_by_config,
        z_full_abs,
        z_full_rel,
        ET_full,
        ET_full_meas,
        p_all_ν_mc_full,
        p_final_ν_full,
        int_dz_E_mc_full,
        ∫dV_E_full,
        boostfactor_full,
        df_ET_full,
        df_bf_analysis_full,
        df_bf_mc_analysis_full,
    )

    println("Saved variation boostfactor results to $(RESULTS_FILE)")end
